# DSMS reference quality, RL learning, and clean-rollout evaluation

This notebook connects reference quality to downstream RL performance for paired GhostGUI and DSMS versions of lunges, crawlstand, and theworm. It produces learning-efficiency tables, learning curves, reference and fidelity figures, a joined evaluation table, a crawl final-transition case study, and a quality-versus-trackability scatter. In the configured files, names ending in gg are GhostGUI and the paired non-gg files are DSMS.

In [ ]:
from __future__ import annotations

import csv
import gzip
import html
import json
import math
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display


def find_repo_root(start: Path | None = None) -> Path:
  start = (start or Path.cwd()).resolve()
  for candidate in (start, *start.parents):
    if (candidate / "pyproject.toml").is_file():
      return candidate
  raise FileNotFoundError("Start Jupyter inside the mjlab repository.")


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / "logs/dsms_analysis"
CACHE_DIR = OUTPUT_DIR / "wandb_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

WANDB_ENTITY = "yitxuenglim-california-institute-of-technology-caltech"
WANDB_PROJECT = "mjlab"
REFRESH_WANDB_CACHE = False
SAVE_FIGURES = True
MAKE_QUALITY_TRACKABILITY_SCATTER = True

FINAL_WINDOW_ITERATIONS = 1_000
SMOOTHING_WINDOW_ITERATIONS = 250
THRESHOLD_HOLD_ITERATIONS = 250
MIN_THRESHOLD_ITERATION = 1_000
POSITION_ERROR_SCALE_RAD = 0.1
VELOCITY_ERROR_SCALE_RAD_S = 1.0
CRAWL_FINAL_STAND_START_S = 10.0
KEYFRAME_PHASES = np.asarray([0.0, 0.25, 0.5, 0.75, 1.0])
END_EFFECTOR_NAMES = (
  "left_ankle_roll_link",
  "right_ankle_roll_link",
  "left_wrist_yaw_link",
  "right_wrist_yaw_link",
)
END_EFFECTOR_CONSTRAINT_TOLERANCE_M = 0.05
DIFFERENTIATION_EDGE_TRIM = 2


@dataclass(frozen=True)
class RunSpec:
  motion: str
  condition: Literal["GhostGUI", "DSMS"]
  run_id: str
  reference_file: Path
  trace_file: Path
  checkpoint_iteration: int

  @property
  def key(self) -> str:
    return f"{self.motion}/{self.condition}"


@dataclass(frozen=True)
class MetricSpec:
  label: str
  key: str
  direction: Literal["higher", "lower"]
  threshold: float
  unit: str


RUNS = (
  RunSpec(
    "Lunges",
    "DSMS",
    "3iimc6dh",
    ROOT / "motion/lunges2.npz",
    ROOT / "logs/lunges2_trace.csv",
    20_000,
  ),
  RunSpec(
    "Lunges",
    "GhostGUI",
    "a6sa5636",
    ROOT / "motion/lunges2gg.npz",
    ROOT / "logs/lunges2gg_trace.csv",
    20_000,
  ),
  RunSpec(
    "Crawlstand",
    "DSMS",
    "emyg51fi",
    ROOT / "motion/crawlstand1.npz",
    ROOT / "logs/crawlstand_trace_30000.csv",
    30_000,
  ),
  RunSpec(
    "Crawlstand",
    "GhostGUI",
    "3vzrknd9",
    ROOT / "motion/crawlstand1gg.npz",
    ROOT / "logs/crawlstandgg_trace.csv",
    20_000,
  ),
  RunSpec(
    "Theworm",
    "DSMS",
    "yw6un3kd",
    ROOT / "motion/theworm2.npz",
    ROOT / "logs/theworm_trace.csv",
    30_000,
  ),
  RunSpec(
    "Theworm",
    "GhostGUI",
    "kq6b014t",
    ROOT / "motion/theworm1gg.npz",
    ROOT / "logs/thewormgg_trace.csv",
    30_000,
  ),
)

METRICS = (
  MetricSpec("Mean reward", "Train/mean_reward", "higher", 30.0, ""),
  MetricSpec(
    "Body-orientation reward", "Episode_Reward/motion_body_ori", "higher", 0.75, ""
  ),
  MetricSpec(
    "Anchor-position error", "Metrics/motion/error_anchor_pos", "lower", 0.12, "m"
  ),
  MetricSpec(
    "Body-orientation error", "Metrics/motion/error_body_rot", "lower", 0.20, "rad"
  ),
  MetricSpec(
    "Joint-position L2 error", "Metrics/motion/error_joint_pos", "lower", 0.90, "rad"
  ),
  MetricSpec(
    "Action-rate penalty", "Episode_Reward/action_rate_l2", "higher", -0.50, ""
  ),
)
LEARNING_FIGURE_METRIC_KEYS = (
  "Train/mean_reward",
  "Episode_Reward/motion_body_ori",
  "Metrics/motion/error_anchor_pos",
  "Episode_Reward/action_rate_l2",
)
MOTIONS = tuple(dict.fromkeys(spec.motion for spec in RUNS))
CONDITIONS = ("GhostGUI", "DSMS")
CONDITION_COLORS = {"GhostGUI": "#3366CC", "DSMS": "#E8752A"}
CONDITION_MARKERS = {"GhostGUI": "o", "DSMS": "s"}
plt.style.use("seaborn-v0_8-whitegrid")
print(f"Repository: {ROOT}")
print(f"Analysis output: {OUTPUT_DIR}")

## 1. Load and cache W&B training histories

The cache makes reruns deterministic and avoids repeatedly downloading histories. Set REFRESH_WANDB_CACHE to True when the remote runs change.

In [ ]:
def cache_path(run_id: str) -> Path:
  return CACHE_DIR / f"{run_id}.json.gz"


def load_wandb_history(spec: RunSpec) -> dict[str, np.ndarray]:
  path = cache_path(spec.run_id)
  if path.is_file() and not REFRESH_WANDB_CACHE:
    with gzip.open(path, "rt", encoding="utf-8") as stream:
      payload = json.load(stream)
    return {key: np.asarray(value, dtype=float) for key, value in payload.items()}

  try:
    import wandb

    api = wandb.Api(timeout=60)
    run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{spec.run_id}")
    keys = ["_step", *(metric.key for metric in METRICS)]
    payload: dict[str, list[float | None]] = {key: [] for key in keys}
    for row in run.scan_history(keys=keys, page_size=1_000):
      for key in keys:
        value = row.get(key)
        payload[key].append(float(value) if value is not None else None)
  except Exception as exc:
    raise RuntimeError(
      f"Could not load W&B run {spec.run_id}. Run 'uv run wandb login', "
      "or restore the cache under logs/dsms_analysis/wandb_cache."
    ) from exc

  with gzip.open(path, "wt", encoding="utf-8") as stream:
    json.dump(payload, stream, separators=(",", ":"))
  return {key: np.asarray(value, dtype=float) for key, value in payload.items()}


histories = {spec.key: load_wandb_history(spec) for spec in RUNS}
print(f"Loaded {len(histories)} histories; cache: {CACHE_DIR}")

## 2. Learning efficiency

AUC is normalized by the common training horizon for each paired motion, so unequal run lengths do not advantage the longer run. Threshold time is the first sustained 250-iteration crossing after iteration 1,000. Final means use each run's last 1,000 iterations.

In [ ]:
def finite_xy(
  history: dict[str, np.ndarray], key: str
) -> tuple[np.ndarray, np.ndarray]:
  x = history["_step"]
  y = history[key]
  valid = np.isfinite(x) & np.isfinite(y)
  order = np.argsort(x[valid])
  return x[valid][order], y[valid][order]


def rolling_curve(
  x: np.ndarray, y: np.ndarray, window: int
) -> tuple[np.ndarray, np.ndarray]:
  if len(y) < 2:
    return x, y
  dx = float(np.median(np.diff(x)))
  samples = max(1, int(round(window / max(dx, 1.0))))
  samples = min(samples, len(y))
  kernel = np.ones(samples) / samples
  smooth = np.convolve(y, kernel, mode="valid")
  return x[samples - 1 :], smooth


def normalized_auc(x: np.ndarray, y: np.ndarray, budget: float) -> float:
  mask = x <= budget
  x_use, y_use = x[mask], y[mask]
  if len(x_use) < 2:
    return math.nan
  if x_use[-1] < budget:
    y_at_budget = float(np.interp(budget, x, y))
    x_use = np.append(x_use, budget)
    y_use = np.append(y_use, y_at_budget)
  span = float(x_use[-1] - x_use[0])
  return float(np.trapezoid(y_use, x_use) / span) if span > 0 else math.nan


def final_window_mean(x: np.ndarray, y: np.ndarray, window: int) -> float:
  return float(np.mean(y[x >= x[-1] - window]))


def iterations_to_threshold(x: np.ndarray, y: np.ndarray, metric: MetricSpec) -> float:
  sx, sy = rolling_curve(x, y, THRESHOLD_HOLD_ITERATIONS)
  crossed = (
    sy >= metric.threshold if metric.direction == "higher" else sy <= metric.threshold
  )
  eligible = np.flatnonzero(crossed & (sx >= MIN_THRESHOLD_ITERATION))
  return float(sx[eligible[0]]) if len(eligible) else math.nan


def save_rows(rows: list[dict[str, Any]], filename: str) -> Path:
  path = OUTPUT_DIR / filename
  if not rows:
    return path
  with path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
  return path


def display_rows(rows: list[dict[str, Any]], digits: int = 4) -> None:
  if not rows:
    display(HTML("<em>No rows</em>"))
    return
  columns = list(rows[0])

  def format_value(value: Any) -> str:
    if isinstance(value, (float, np.floating)):
      return "—" if not np.isfinite(value) else f"{value:.{digits}f}"
    return html.escape(str(value))

  header = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
  body = "".join(
    "<tr>"
    + "".join(f"<td>{format_value(row[column])}</td>" for column in columns)
    + "</tr>"
    for row in rows
  )
  display(
    HTML(
      f"<div style='overflow-x:auto'><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    )
  )


common_budgets = {
  motion: min(
    finite_xy(histories[spec.key], METRICS[0].key)[0][-1]
    for spec in RUNS
    if spec.motion == motion
  )
  for motion in MOTIONS
}
learning_rows: list[dict[str, Any]] = []
for spec in RUNS:
  for metric in METRICS:
    x, y = finite_xy(histories[spec.key], metric.key)
    learning_rows.append(
      {
        "motion": spec.motion,
        "condition": spec.condition,
        "metric": metric.label,
        "direction": metric.direction,
        "threshold": metric.threshold,
        "common_budget": common_budgets[spec.motion],
        "auc_common_budget": normalized_auc(x, y, common_budgets[spec.motion]),
        "final_1k_mean": final_window_mean(x, y, FINAL_WINDOW_ITERATIONS),
        "iterations_to_threshold": iterations_to_threshold(x, y, metric),
        "run_id": spec.run_id,
      }
    )
learning_csv = save_rows(learning_rows, "learning_efficiency.csv")
display_rows(learning_rows)
print(f"Saved {learning_csv}")

In [ ]:
metric_by_key = {metric.key: metric for metric in METRICS}
fig, axes = plt.subplots(
  len(LEARNING_FIGURE_METRIC_KEYS), len(MOTIONS), figsize=(15, 12), sharex="col"
)
for column, motion in enumerate(MOTIONS):
  for row, metric_key in enumerate(LEARNING_FIGURE_METRIC_KEYS):
    ax = axes[row, column]
    metric = metric_by_key[metric_key]
    for spec in RUNS:
      if spec.motion != motion:
        continue
      x, y = finite_xy(histories[spec.key], metric_key)
      sx, sy = rolling_curve(x, y, SMOOTHING_WINDOW_ITERATIONS)
      ax.plot(sx, sy, color=CONDITION_COLORS[spec.condition], label=spec.condition)
    ax.axhline(metric.threshold, color="0.25", linestyle="--", linewidth=1)
    ax.axvline(common_budgets[motion], color="0.55", linestyle=":", linewidth=1)
    if row == 0:
      ax.set_title(motion)
    if column == 0:
      ax.set_ylabel(metric.label + (f" [{metric.unit}]" if metric.unit else ""))
    if row == len(LEARNING_FIGURE_METRIC_KEYS) - 1:
      ax.set_xlabel("Training iteration")
axes[0, -1].legend(frameon=True)
fig.suptitle("Learning curves (250-iteration rolling mean)", fontsize=15)
fig.tight_layout()
learning_figure_path = OUTPUT_DIR / "learning_curves.png"
if SAVE_FIGURES:
  fig.savefig(learning_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {learning_figure_path}")

## 3. Reference quality and DSMS fidelity

Reference acceleration and jerk use the stored joint velocity and its numerical derivatives. Fidelity compares DSMS with GhostGUI at five normalized-motion keyframes. End-effector positions are pelvis-relative, removing global translation. The constraint residual reported here is an observable proxy: end-effector deviation beyond the configured 5 cm tolerance. It is not the residual of an unlogged DSMS optimizer constraint.

In [ ]:
@dataclass(frozen=True)
class MotionData:
  fps: float
  joint_pos: np.ndarray
  joint_vel: np.ndarray
  body_pos_w: np.ndarray

  @property
  def duration_s(self) -> float:
    return (len(self.joint_pos) - 1) / self.fps


@dataclass(frozen=True)
class TraceData:
  time_s: np.ndarray
  q_ref: np.ndarray
  q_robot: np.ndarray
  v_ref: np.ndarray
  v_robot: np.ndarray
  scalars: dict[str, np.ndarray]

  @property
  def q_rmse(self) -> np.ndarray:
    return np.sqrt(np.mean((self.q_robot - self.q_ref) ** 2, axis=1))

  @property
  def v_rmse(self) -> np.ndarray:
    return np.sqrt(np.mean((self.v_robot - self.v_ref) ** 2, axis=1))

  @property
  def state_rmse(self) -> np.ndarray:
    q_scaled = (self.q_robot - self.q_ref) / POSITION_ERROR_SCALE_RAD
    v_scaled = (self.v_robot - self.v_ref) / VELOCITY_ERROR_SCALE_RAD_S
    return np.sqrt(np.mean(np.concatenate((q_scaled, v_scaled), axis=1) ** 2, axis=1))


def load_motion(path: Path) -> MotionData:
  with np.load(path) as data:
    return MotionData(
      fps=float(np.asarray(data["fps"]).reshape(-1)[0]),
      joint_pos=np.asarray(data["joint_pos"], dtype=float),
      joint_vel=np.asarray(data["joint_vel"], dtype=float),
      body_pos_w=np.asarray(data["body_pos_w"], dtype=float),
    )


def load_trace(path: Path) -> TraceData:
  vector_columns = (
    "reference_joint_pos",
    "actual_joint_pos",
    "reference_joint_vel",
    "actual_joint_vel",
  )
  scalar_columns = (
    "anchor_position_error",
    "anchor_rotation_error",
    "body_position_error",
    "body_rotation_error",
    "total_reward_rate",
    "reward/action_rate_l2",
  )
  vectors: dict[str, list[list[float]]] = {key: [] for key in vector_columns}
  scalars: dict[str, list[float]] = {key: [] for key in scalar_columns}
  time_s: list[float] = []
  with path.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    missing = set(vector_columns) - set(reader.fieldnames or ())
    if missing:
      raise ValueError(f"{path.name} is missing trace columns: {sorted(missing)}")
    for row in reader:
      time_s.append(float(row["sim_time_s"]))
      for key in vector_columns:
        vectors[key].append(json.loads(row[key]))
      for key in scalar_columns:
        value = row.get(key, "")
        scalars[key].append(float(value) if value else math.nan)
  return TraceData(
    time_s=np.asarray(time_s),
    q_ref=np.asarray(vectors["reference_joint_pos"]),
    q_robot=np.asarray(vectors["actual_joint_pos"]),
    v_ref=np.asarray(vectors["reference_joint_vel"]),
    v_robot=np.asarray(vectors["actual_joint_vel"]),
    scalars={key: np.asarray(value) for key, value in scalars.items()},
  )


def sample_by_phase(values: np.ndarray, phases: np.ndarray) -> np.ndarray:
  sample_positions = phases * (len(values) - 1)
  left = np.floor(sample_positions).astype(int)
  right = np.ceil(sample_positions).astype(int)
  fraction = sample_positions - left
  shape = (len(fraction),) + (1,) * (values.ndim - 1)
  return values[left] * (1.0 - fraction.reshape(shape)) + values[
    right
  ] * fraction.reshape(shape)


motions = {spec.key: load_motion(spec.reference_file) for spec in RUNS}
traces = {spec.key: load_trace(spec.trace_file) for spec in RUNS}
robot_xml = ROOT / "src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml"
xml_root = ET.parse(robot_xml).getroot()
body_names = [
  body.attrib["name"] for body in xml_root.findall(".//body") if "name" in body.attrib
]
if len(body_names) != next(iter(motions.values())).body_pos_w.shape[1]:
  raise ValueError("G1 XML body order does not match the reference body arrays.")
body_index = {name: index for index, name in enumerate(body_names)}
pelvis_index = body_index["pelvis"]
end_effector_indices = [body_index[name] for name in END_EFFECTOR_NAMES]
print(f"Loaded {len(motions)} references and {len(traces)} rollout traces")

In [ ]:
def rms(values: np.ndarray) -> float:
  return float(np.sqrt(np.mean(np.asarray(values, dtype=float) ** 2)))


def reference_quality(motion: MotionData) -> dict[str, float]:
  acceleration = np.gradient(motion.joint_vel, 1.0 / motion.fps, axis=0)
  jerk = np.gradient(acceleration, 1.0 / motion.fps, axis=0)
  trim = DIFFERENTIATION_EDGE_TRIM
  acceleration = (
    acceleration[trim:-trim] if len(acceleration) > 2 * trim else acceleration
  )
  jerk = jerk[trim:-trim] if len(jerk) > 2 * trim else jerk
  acceleration_frame_rms = np.sqrt(np.mean(acceleration**2, axis=1))
  jerk_frame_rms = np.sqrt(np.mean(jerk**2, axis=1))
  return {
    "frames": len(motion.joint_pos),
    "fps": motion.fps,
    "duration_s": motion.duration_s,
    "acceleration_rms_rad_s2": rms(acceleration),
    "acceleration_p95_frame_rms_rad_s2": float(
      np.percentile(acceleration_frame_rms, 95)
    ),
    "jerk_rms_rad_s3": rms(jerk),
    "jerk_p95_frame_rms_rad_s3": float(np.percentile(jerk_frame_rms, 95)),
  }


reference_rows: list[dict[str, Any]] = []
reference_lookup: dict[str, dict[str, float]] = {}
for spec in RUNS:
  values = reference_quality(motions[spec.key])
  reference_lookup[spec.key] = values
  reference_rows.append({"motion": spec.motion, "condition": spec.condition, **values})

fidelity_rows: list[dict[str, Any]] = []
fidelity_lookup: dict[str, dict[str, float]] = {}
for motion_name in MOTIONS:
  baseline = motions[f"{motion_name}/GhostGUI"]
  dsms = motions[f"{motion_name}/DSMS"]
  q_delta = sample_by_phase(dsms.joint_pos, KEYFRAME_PHASES) - sample_by_phase(
    baseline.joint_pos, KEYFRAME_PHASES
  )
  baseline_body = sample_by_phase(baseline.body_pos_w, KEYFRAME_PHASES)
  dsms_body = sample_by_phase(dsms.body_pos_w, KEYFRAME_PHASES)
  baseline_ee = (
    baseline_body[:, end_effector_indices] - baseline_body[:, [pelvis_index]]
  )
  dsms_ee = dsms_body[:, end_effector_indices] - dsms_body[:, [pelvis_index]]
  ee_deviation = np.linalg.norm(dsms_ee - baseline_ee, axis=-1)
  residual = np.maximum(ee_deviation - END_EFFECTOR_CONSTRAINT_TOLERANCE_M, 0.0)
  values = {
    "keyframe_joint_rmse_rad": rms(q_delta),
    "keyframe_ee_mean_deviation_m": float(np.mean(ee_deviation)),
    "keyframe_ee_max_deviation_m": float(np.max(ee_deviation)),
    "endpoint_ee_mean_deviation_m": float(np.mean(ee_deviation[-1])),
    "timing_change_s": dsms.duration_s - baseline.duration_s,
    "timing_change_percent": 100.0 * (dsms.duration_s / baseline.duration_s - 1.0),
    "constraint_residual_proxy_rms_m": rms(residual),
    "constraint_violation_rate": float(
      np.mean(ee_deviation > END_EFFECTOR_CONSTRAINT_TOLERANCE_M)
    ),
  }
  fidelity_lookup[f"{motion_name}/DSMS"] = values
  fidelity_rows.append({"motion": motion_name, **values})

reference_csv = save_rows(reference_rows, "reference_quality.csv")
fidelity_csv = save_rows(fidelity_rows, "dsms_fidelity.csv")
display(HTML("<h4>Reference quality</h4>"))
display_rows(reference_rows)
display(HTML("<h4>DSMS fidelity to GhostGUI</h4>"))
display_rows(fidelity_rows)
print(f"Saved {reference_csv} and {fidelity_csv}")

In [ ]:
quality_metrics = (
  ("acceleration_rms_rad_s2", "Joint acceleration RMS [rad/s²]"),
  ("jerk_rms_rad_s3", "Joint jerk RMS [rad/s³]"),
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (metric_key, label) in zip(axes, quality_metrics, strict=True):
  for index, motion_name in enumerate(MOTIONS):
    before = reference_lookup[f"{motion_name}/GhostGUI"][metric_key]
    after = reference_lookup[f"{motion_name}/DSMS"][metric_key]
    ax.plot(
      [0, 1],
      [before, after],
      marker=CONDITION_MARKERS["DSMS"],
      color=plt.cm.tab10(index),
      linewidth=2,
      label=motion_name,
    )
  ax.set_xticks([0, 1], CONDITIONS)
  ax.set_yscale("log")
  ax.set_ylabel(label)
  ax.set_title(label.split(" [")[0])
axes[-1].legend(title="Motion", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Paired reference quality: GhostGUI to DSMS")
fig.tight_layout()
reference_figure_path = OUTPUT_DIR / "reference_quality_paired.png"
if SAVE_FIGURES:
  fig.savefig(reference_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {reference_figure_path}")

## 4. Final evaluation table

Each row joins reference quality, clean one-environment rollout tracking, final training metrics, and DSMS fidelity. The combined state error is dimensionless: position error is divided by 0.1 rad and velocity error by 1 rad/s before taking RMSE across all position and velocity elements.

In [ ]:
def finite_mean(values: np.ndarray) -> float:
  values = np.asarray(values, dtype=float)
  return float(np.mean(values[np.isfinite(values)]))


def training_final(spec: RunSpec, metric_key: str) -> float:
  x, y = finite_xy(histories[spec.key], metric_key)
  return final_window_mean(x, y, FINAL_WINDOW_ITERATIONS)


evaluation_rows: list[dict[str, Any]] = []
for spec in RUNS:
  trace = traces[spec.key]
  quality = reference_lookup[spec.key]
  fidelity = fidelity_lookup.get(spec.key, {})
  evaluation_rows.append(
    {
      "motion": spec.motion,
      "condition": spec.condition,
      "checkpoint_iteration": spec.checkpoint_iteration,
      "reference_duration_s": quality["duration_s"],
      "reference_acceleration_rms_rad_s2": quality["acceleration_rms_rad_s2"],
      "reference_jerk_rms_rad_s3": quality["jerk_rms_rad_s3"],
      "tracking_position_rmse_rad": rms(trace.q_robot - trace.q_ref),
      "tracking_velocity_rmse_rad_s": rms(trace.v_robot - trace.v_ref),
      "tracking_state_rmse_scaled": rms(trace.state_rmse),
      "tracking_state_p95_scaled": float(np.percentile(trace.state_rmse, 95)),
      "rollout_anchor_position_error_m": finite_mean(
        trace.scalars["anchor_position_error"]
      ),
      "rollout_body_orientation_error_rad": finite_mean(
        trace.scalars["body_rotation_error"]
      ),
      "rollout_reward_rate": finite_mean(trace.scalars["total_reward_rate"]),
      "rollout_action_rate_penalty": finite_mean(
        trace.scalars["reward/action_rate_l2"]
      ),
      "training_final_1k_reward": training_final(spec, "Train/mean_reward"),
      "training_final_1k_body_ori_reward": training_final(
        spec, "Episode_Reward/motion_body_ori"
      ),
      "training_final_1k_action_rate_penalty": training_final(
        spec, "Episode_Reward/action_rate_l2"
      ),
      "fidelity_keyframe_joint_rmse_rad": fidelity.get("keyframe_joint_rmse_rad", 0.0),
      "fidelity_ee_mean_deviation_m": fidelity.get("keyframe_ee_mean_deviation_m", 0.0),
      "fidelity_timing_change_percent": fidelity.get("timing_change_percent", 0.0),
      "constraint_residual_proxy_rms_m": fidelity.get(
        "constraint_residual_proxy_rms_m", 0.0
      ),
    }
  )
evaluation_csv = save_rows(evaluation_rows, "final_evaluation.csv")
display_rows(evaluation_rows)
print(f"Saved {evaluation_csv}")

## 5. Crawlstand case study

The upper panels show optimization history. The clean-rollout panel shows the scaled joint-state error and shades the configured final stand/transition phase from 10 s onward. Change CRAWL_FINAL_STAND_START_S if the semantic phase boundary changes.

In [ ]:
crawl_training_metrics = (
  ("Episode_Reward/motion_body_ori", "Body-orientation reward"),
  ("Metrics/motion/error_anchor_pos", "Anchor-position error [m]"),
)
fig, axes = plt.subplots(3, 1, figsize=(11, 10))
for ax, (metric_key, label) in zip(axes[:2], crawl_training_metrics, strict=True):
  for condition in CONDITIONS:
    key = f"Crawlstand/{condition}"
    x, y = finite_xy(histories[key], metric_key)
    sx, sy = rolling_curve(x, y, SMOOTHING_WINDOW_ITERATIONS)
    ax.plot(sx, sy, color=CONDITION_COLORS[condition], label=condition)
  ax.axvline(common_budgets["Crawlstand"], color="0.45", linestyle=":")
  ax.set_ylabel(label)
  ax.set_xlabel("Training iteration")
axes[0].legend()
for condition in CONDITIONS:
  trace = traces[f"Crawlstand/{condition}"]
  axes[2].plot(
    trace.time_s, trace.state_rmse, color=CONDITION_COLORS[condition], label=condition
  )
axes[2].axvspan(
  CRAWL_FINAL_STAND_START_S,
  max(trace.time_s[-1] for trace in traces.values()),
  color="#D8B365",
  alpha=0.22,
  label="Final stand / transition",
)
axes[2].axvline(CRAWL_FINAL_STAND_START_S, color="#8C510A", linestyle="--")
axes[2].set_xlabel("Clean-rollout time [s]")
axes[2].set_ylabel("Scaled joint-state RMSE")
axes[2].legend()
fig.suptitle("Crawlstand: learning dynamics and final-transition tracking", fontsize=15)
fig.tight_layout()
crawl_figure_path = OUTPUT_DIR / "crawl_phase_case_study.png"
if SAVE_FIGURES:
  fig.savefig(crawl_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {crawl_figure_path}")

## 6. Optional quality-versus-trackability view

This scatter is descriptive only: six points are too few for a strong inferential claim, and the paired motions differ in fidelity and duration as well as smoothness.

In [ ]:
if MAKE_QUALITY_TRACKABILITY_SCATTER:
  jerk_values = np.asarray(
    [row["reference_jerk_rms_rad_s3"] for row in evaluation_rows]
  )
  tracking_values = np.asarray(
    [row["tracking_state_rmse_scaled"] for row in evaluation_rows]
  )
  correlation = float(np.corrcoef(np.log10(jerk_values), tracking_values)[0, 1])
  fig, ax = plt.subplots(figsize=(8, 5.5))
  for row in evaluation_rows:
    ax.scatter(
      row["reference_jerk_rms_rad_s3"],
      row["tracking_state_rmse_scaled"],
      color=CONDITION_COLORS[row["condition"]],
      marker=CONDITION_MARKERS[row["condition"]],
      s=70,
    )
    ax.annotate(
      f"{row['motion']} {row['condition']}",
      (row["reference_jerk_rms_rad_s3"], row["tracking_state_rmse_scaled"]),
      xytext=(5, 5),
      textcoords="offset points",
      fontsize=8,
    )
  ax.set_xscale("log")
  ax.set_xlabel("Reference joint jerk RMS [rad/s³]")
  ax.set_ylabel("Clean-rollout scaled state RMSE")
  ax.set_title(
    f"Reference quality vs trackability (descriptive r = {correlation:.2f}, n = 6)"
  )
  scatter_figure_path = OUTPUT_DIR / "quality_vs_trackability.png"
  if SAVE_FIGURES:
    fig.savefig(scatter_figure_path, dpi=180, bbox_inches="tight")
  plt.show()
  print(f"Saved {scatter_figure_path}")

## 7. All clean-rollout traces

This section is independent of the six W&B learning runs above. It loads every available local trace, including standard, custom-reward, and contact-rich policies. Files ending in gg are labeled GhostGUI; non-gg references are labeled DSMS. Task variants are kept explicit so a reward/configuration change is not mistaken for a reference-method change.

In [ ]:
@dataclass(frozen=True)
class TracePlotSpec:
  key: str
  family: str
  condition: Literal["GhostGUI", "DSMS"]
  training_config: str
  checkpoint_iteration: int
  trace_file: Path

  @property
  def curve_label(self) -> str:
    checkpoint = f"{self.checkpoint_iteration / 1000:g}k"
    return f"{self.condition}, {self.training_config}, {checkpoint}"


TRACE_SPECS = (
  TracePlotSpec(
    "crawl-dsms-standard-20k",
    "Crawlstand",
    "DSMS",
    "Standard",
    20_000,
    ROOT / "logs/crawlstand_trace.csv",
  ),
  TracePlotSpec(
    "crawl-dsms-standard-30k",
    "Crawlstand",
    "DSMS",
    "Standard",
    30_000,
    ROOT / "logs/crawlstand_trace_30000.csv",
  ),
  TracePlotSpec(
    "crawl-ghostgui-standard-20k",
    "Crawlstand",
    "GhostGUI",
    "Standard",
    20_000,
    ROOT / "logs/crawlstandgg_trace.csv",
  ),
  TracePlotSpec(
    "crawl-dsms-contact-rich-30k",
    "Crawlstand",
    "DSMS",
    "Contact-rich",
    30_000,
    ROOT / "logs/crawlstand_contactrich_trace.csv",
  ),
  TracePlotSpec(
    "lunges-dsms-standard-20k",
    "Lunges",
    "DSMS",
    "Standard",
    20_000,
    ROOT / "logs/lunges_trace.csv",
  ),
  TracePlotSpec(
    "lunges2-dsms-standard-20k",
    "Lunges2",
    "DSMS",
    "Standard",
    20_000,
    ROOT / "logs/lunges2_trace.csv",
  ),
  TracePlotSpec(
    "lunges2-ghostgui-standard-20k",
    "Lunges2",
    "GhostGUI",
    "Standard",
    20_000,
    ROOT / "logs/lunges2gg_trace.csv",
  ),
  TracePlotSpec(
    "lunges2-dsms-custom-30k",
    "Lunges2",
    "DSMS",
    "Custom",
    30_000,
    ROOT / "logs/lunges2_custom_trace.csv",
  ),
  TracePlotSpec(
    "theworm-dsms-standard-30k",
    "Theworm",
    "DSMS",
    "Standard",
    30_000,
    ROOT / "logs/theworm_trace.csv",
  ),
  TracePlotSpec(
    "theworm-ghostgui-standard-30k",
    "Theworm",
    "GhostGUI",
    "Standard",
    30_000,
    ROOT / "logs/thewormgg_trace.csv",
  ),
  TracePlotSpec(
    "worm1-dsms-standard-30k",
    "Worm1",
    "DSMS",
    "Standard",
    30_000,
    ROOT / "logs/worm1_trace.csv",
  ),
  TracePlotSpec(
    "worm1-ghostgui-standard-30k",
    "Worm1",
    "GhostGUI",
    "Standard",
    30_000,
    ROOT / "logs/worm1gg_trace.csv",
  ),
  TracePlotSpec(
    "getup-dsms-standard-30k",
    "Getup",
    "DSMS",
    "Standard",
    30_000,
    ROOT / "logs/getup_trace.csv",
  ),
  TracePlotSpec(
    "getup-dsms-custom-4.5k",
    "Getup",
    "DSMS",
    "Custom",
    4_500,
    ROOT / "logs/getup_custom_4500_trace.csv",
  ),
  TracePlotSpec(
    "kneel-dsms-standard-20k",
    "Kneel",
    "DSMS",
    "Standard",
    20_000,
    ROOT / "logs/kneel_trace.csv",
  ),
  TracePlotSpec(
    "slidefloat-dsms-standard-20k",
    "Slidefloat",
    "DSMS",
    "Standard",
    20_000,
    ROOT / "logs/slidefloat_trace.csv",
  ),
  TracePlotSpec(
    "facedown-dsms-custom-30k",
    "Facedown",
    "DSMS",
    "Custom",
    30_000,
    ROOT / "logs/facedown_trace.csv",
  ),
  TracePlotSpec(
    "facedown-ghostgui-custom-30k",
    "Facedown",
    "GhostGUI",
    "Custom",
    30_000,
    ROOT / "logs/facedowngg_trace.csv",
  ),
  TracePlotSpec(
    "pushup-dsms-custom-30k",
    "Pushup",
    "DSMS",
    "Custom",
    30_000,
    ROOT / "logs/pushup_trace.csv",
  ),
  TracePlotSpec(
    "pushup-ghostgui-custom-30k",
    "Pushup",
    "GhostGUI",
    "Custom",
    30_000,
    ROOT / "logs/pushupgg_trace.csv",
  ),
  TracePlotSpec(
    "squat-dsms-custom-30k",
    "Squat",
    "DSMS",
    "Custom",
    30_000,
    ROOT / "logs/squat_trace.csv",
  ),
  TracePlotSpec(
    "squat-ghostgui-custom-30k",
    "Squat",
    "GhostGUI",
    "Custom",
    30_000,
    ROOT / "logs/squatgg_trace.csv",
  ),
)

missing_trace_files = [
  spec.trace_file for spec in TRACE_SPECS if not spec.trace_file.is_file()
]
if missing_trace_files:
  missing = "\n".join(f"- {path}" for path in missing_trace_files)
  raise FileNotFoundError(
    f"Generate the missing trace files before running this section:\n{missing}"
  )

all_traces = {spec.key: load_trace(spec.trace_file) for spec in TRACE_SPECS}
print(f"Loaded all {len(all_traces)} clean-rollout traces")

In [ ]:
def trace_metric_series(trace: TraceData, metric: str) -> np.ndarray:
  if metric == "position":
    return trace.q_rmse
  if metric == "velocity":
    return trace.v_rmse
  if metric == "state":
    return trace.state_rmse
  raise KeyError(metric)


all_trace_rows: list[dict[str, Any]] = []
for spec in TRACE_SPECS:
  trace = all_traces[spec.key]
  all_trace_rows.append(
    {
      "trace": spec.key,
      "motion": spec.family,
      "condition": spec.condition,
      "training_config": spec.training_config,
      "checkpoint_iteration": spec.checkpoint_iteration,
      "duration_s": float(trace.time_s[-1] - trace.time_s[0]),
      "position_rmse_rad": rms(trace.q_robot - trace.q_ref),
      "position_p95_rad": float(np.percentile(trace.q_rmse, 95)),
      "velocity_rmse_rad_s": rms(trace.v_robot - trace.v_ref),
      "velocity_p95_rad_s": float(np.percentile(trace.v_rmse, 95)),
      "state_rmse_scaled": rms(trace.state_rmse),
      "state_p95_scaled": float(np.percentile(trace.state_rmse, 95)),
      "anchor_position_error_m": finite_mean(trace.scalars["anchor_position_error"]),
      "body_orientation_error_rad": finite_mean(trace.scalars["body_rotation_error"]),
      "reward_rate": finite_mean(trace.scalars["total_reward_rate"]),
      "action_rate_penalty": finite_mean(trace.scalars["reward/action_rate_l2"]),
    }
  )
all_trace_csv = save_rows(all_trace_rows, "all_trace_evaluation.csv")
display_rows(all_trace_rows)
print(f"Saved {all_trace_csv}")

summary_metrics = (
  ("position_rmse_rad", "position_p95_rad", "Joint-position error [rad]"),
  ("velocity_rmse_rad_s", "velocity_p95_rad_s", "Joint-velocity error [rad/s]"),
  ("state_rmse_scaled", "state_p95_scaled", "Scaled joint-state error"),
)
y = np.arange(len(all_trace_rows))
fig, axes = plt.subplots(
  1, 3, figsize=(18, max(9, 0.42 * len(all_trace_rows))), sharey=True
)
for ax, (mean_key, p95_key, label) in zip(axes, summary_metrics, strict=True):
  means = np.asarray([row[mean_key] for row in all_trace_rows])
  p95 = np.asarray([row[p95_key] for row in all_trace_rows])
  colors = [CONDITION_COLORS[row["condition"]] for row in all_trace_rows]
  ax.barh(y, means, color=colors, alpha=0.82)
  ax.scatter(p95, y, marker="|", s=90, color="black", label="Framewise p95")
  ax.set_xlabel(label)
  ax.invert_yaxis()
axes[0].set_yticks(y, [row["trace"] for row in all_trace_rows], fontsize=8)
for condition in CONDITIONS:
  axes[-1].plot([], [], color=CONDITION_COLORS[condition], linewidth=8, label=condition)
axes[-1].legend(loc="lower right")
fig.suptitle(
  "All clean rollouts: pooled RMSE bars and framewise p95 markers", fontsize=15
)
fig.tight_layout()
all_trace_summary_path = OUTPUT_DIR / "all_trace_rmse_summary.png"
if SAVE_FIGURES:
  fig.savefig(all_trace_summary_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {all_trace_summary_path}")

In [ ]:
trace_curve_metrics = (
  ("position", "Joint-position RMSE [rad]", "all_traces_position_rmse.png"),
  ("velocity", "Joint-velocity RMSE [rad/s]", "all_traces_velocity_rmse.png"),
  ("state", "Scaled joint-state RMSE", "all_traces_state_rmse.png"),
)
training_line_styles = {"Standard": "-", "Custom": "--", "Contact-rich": ":"}
trace_families = tuple(dict.fromkeys(spec.family for spec in TRACE_SPECS))
ncols = 3
nrows = math.ceil(len(trace_families) / ncols)

for metric_key, ylabel, filename in trace_curve_metrics:
  fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.8 * nrows), squeeze=False)
  for ax, family in zip(axes.flat, trace_families, strict=False):
    for spec in TRACE_SPECS:
      if spec.family != family:
        continue
      series = trace_metric_series(all_traces[spec.key], metric_key)
      phase = np.linspace(0.0, 100.0, len(series))
      ax.plot(
        phase,
        series,
        color=CONDITION_COLORS[spec.condition],
        linestyle=training_line_styles[spec.training_config],
        linewidth=1.7,
        label=spec.curve_label,
      )
    ax.set_title(family)
    ax.set_xlabel("Motion phase [%]")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
  for ax in axes.flat[len(trace_families) :]:
    ax.set_visible(False)
  fig.suptitle(f"All clean rollouts: {ylabel}", fontsize=15)
  fig.tight_layout()
  figure_path = OUTPUT_DIR / filename
  if SAVE_FIGURES:
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
  plt.show()
  print(f"Saved {figure_path}")

## Interpretation guardrails and generated outputs

- Compare AUC only within a motion: it uses that pair's common training budget.
- The final evaluation uses the selected converged checkpoints, so Crawlstand DSMS at 30k versus GhostGUI at 20k is a performance comparison, not a sample-efficiency comparison.
- A smoother DSMS reference is not automatically more faithful or more trackable. Read the fidelity, timing, and rollout columns together.
- The end-effector constraint value is a measurable proxy, not a direct DSMS solver residual. Export true optimizer residuals if they are needed for a formal constraint claim.
- The all-trace section includes every available clean rollout, while W&B learning-efficiency analysis remains limited to the six explicitly configured W&B runs.
- Tables and publication-ready PNGs are saved under logs/dsms_analysis.